---
title: Visualisasi data xarray dari Open Data Cube
short_title: Visualisasi
subject: Panduan Pemula
subtitle: Visualisasi Dataset dan DataArray xarray yang dihasilkan oleh dc.load.
description: Visualisasi Dataset dan DataArray xarray yang dihasilkan oleh dc.load.
keywords:
  - open-data-cube
  - odc
  - xarray
  - plotting
  - beginner-guide
---

Notebook ini menjelaskan cara memvisualisasikan data xarray dari `dc.load`, mulai dari satu band hingga komposit warna alami dan facet grid berisi beberapa panel yang dapat dibandingkan.[^edits]

[^edits]: Notebook tutorial diperbarui secara otomatis sehingga perubahan dapat tertimpa saat pembaruan.
    Simpan salinan kerja dalam file terpisah agar perubahan tetap tersimpan.

## A. Tujuan

- Menampilkan satu band sebagai citra dengan colormap
- Menyusun band merah, hijau, dan biru menjadi komposit warna alami
- Membandingkan satu band untuk beberapa tahun dalam deretan panel

## B. Memuat Data Sampel

Muat dataset kecil yang berisi empat band untuk 2024 dan 2025.

In [ ]:
from datacube import Datacube

dc = Datacube(app="plotting")

query = {
    "product": "s2_geomad_annual",
    "x": (98.80, 98.90),
    "y": (2.65, 2.55),
    "time": ("2024", "2025"),
    "measurements": ["red", "green", "blue", "nir"],
    "output_crs": "EPSG:32647",
    "resolution": (-30, 30),
}

ds = dc.load(**query)

## C. Visualisasi

Plot mengubah array nilai piksel menjadi citra sehingga pola spasial dan nilai yang tidak biasa lebih mudah dikenali.
Di Jupyter notebook, hasilnya muncul tepat di bawah kode sehingga setiap pilihan dalam kode dapat langsung dikaitkan dengan citra yang dihasilkan.

Satu panel citra memerlukan dua dimensi spasial.
`xarray.Dataset` yang dimuat juga memuat beberapa band dan tahun, jadi susunan datanya perlu disesuaikan dengan informasi yang ingin ditampilkan.

## D. Citra Satu Band

Mulailah dengan band merah untuk 2024.
Setelah satu band dan satu tahun dipilih, hasilnya berupa DataArray 2D dengan dimensi spasial `y` dan `x` yang dapat digambar sebagai citra oleh xarray.

In [ ]:
ds.red.isel(time=0).plot()

Kode tersebut merangkai tiga operasi.
Pertama, `ds.red` memilih band merah.
Lalu `.isel(time=0)` memilih tahun pertama berdasarkan urutan data, yaitu 2024.
Terakhir, `.plot()` menggambar nilai 2D dan menambahkan bilah warna yang menghubungkan setiap warna dengan nilai piksel.

Argumen `cmap` mengatur colormap untuk menampilkan nilai tersebut.
Matplotlib menyediakan berbagai pilihan bawaan dalam [referensi colormap matplotlib](https://matplotlib.org/stable/gallery/color/colormap_reference.html).
Colormap berurutan cocok digunakan karena nilai rendah dan tinggi ditampilkan dalam satu skala yang teratur.
Sebagai contoh, `"Reds"` menggunakan gradasi merah dari muda ke tua:

In [ ]:
ds.red.isel(time=0).plot(cmap="Reds")

## E. Komposit Warna

Komposit warna alami menempatkan pengukuran merah, hijau, dan biru pada kanal tampilan yang bersesuaian.
Sebelum divisualisasikan, ketiga band tersebut perlu digabungkan dalam urutan yang tepat ke dalam satu DataArray.

In [ ]:
rgb = ds[["red", "green", "blue"]].to_array(dim="band").isel(time=0)
rgb

Pemilihan `ds[["red", "green", "blue"]]` menghasilkan Dataset yang lebih kecil dengan urutan band merah, hijau, lalu biru.
Pemanggilan `.to_array(dim="band")` menumpuk ketiga variabel data itu pada dimensi baru bernama `band`.
Argumen `dim` bersifat opsional; jika tidak diberikan, xarray menamai dimensi baru tersebut `"variable"`.
Setelah itu, `.isel(time=0)` memilih data 2024.

Hasilnya disimpan dalam variabel Python `rgb` dengan dimensi `(band, y, x)`.
Nama `rgb` memperjelas kegunaan variabel bagi pembaca.
xarray menentukan cara membuat plot berdasarkan dimensi data dan metode yang dipilih.
Urutan tiga elemen pada dimensi `band` memasangkannya dengan kanal tampilan merah, hijau, dan biru.

Pemanggilan metode umum `.plot()` pada DataArray 3D ini menghasilkan tampilan yang berbeda:

In [ ]:
rgb.plot()

Hasilnya berupa histogram dari seluruh nilai.
Pada DataArray satu band yang berbentuk 2D, xarray dapat menggunakan `y` dan `x` sebagai bidang citra.
DataArray `rgb` memiliki satu dimensi tambahan, yaitu `band`, sehingga metode plot umum memilih histogram.

Gunakan `.plot.imshow()` ketika dimensi tambahan tersebut memuat kanal warna:

In [ ]:
rgb.plot.imshow(vmin=0, vmax=3000)

Pada plot ini, `.plot.imshow()` membaca dimensi `band` yang berukuran tiga sebagai kanal merah, hijau, dan biru, sedangkan `y` dan `x` menjadi bidang citra.

Argumen `vmin` dan `vmax` menentukan rentang tampilan untuk setiap kanal.
Nilai yang sama dengan atau lebih kecil dari `vmin` diberi intensitas nol pada kanal tersebut, sedangkan nilai yang sama dengan atau lebih besar dari `vmax` diberi intensitas penuh.
Pengaturan ini hanya mengubah tampilan citra, dan nilai yang tersimpan dalam `rgb` tetap sama.

Rentang yang sesuai bergantung pada produk.
Produk `s2_geomad_annual` dalam latihan ini berasal dari reflektansi permukaan Sentinel-2, yang umum ditampilkan dengan rentang 0 hingga 3000.
Reflektansi permukaan Landsat umumnya menggunakan rentang 0 hingga 7500.
Jalankan plot tanpa `vmin` dan `vmax` untuk melihat pengaruh skala otomatis xarray terhadap tampilan komposit.

## F. Facet Grid

Facet grid menempatkan beberapa plot yang berkaitan dalam panel terpisah dengan aturan tampilan yang sama.
Susunan ini memudahkan perbandingan citra band merah untuk 2024 dan 2025 secara berdampingan.

In [ ]:
ds.red.plot(col="time", cmap="Reds", vmin=0, vmax=3000)

Argumen `col="time"` membuat satu panel untuk setiap nilai pada koordinat `time`, dan setiap panel berisi citra 2D `(y, x)`.
Pengaturan `cmap`, `vmin`, dan `vmax` yang sama berlaku pada semua panel.
Dengan skala yang tetap, perbedaan warna antartahun mencerminkan perbedaan nilai data.

Facet juga dapat digunakan untuk memisahkan setiap elemen pada dimensi `band`:

In [ ]:
rgb.plot(col="band", vmin=0, vmax=3000)

Plot ini berisi satu panel 2D untuk masing-masing band merah, hijau, dan biru.
Ketiga panel menampilkan band penyusun secara terpisah sehingga perbedaannya lebih mudah diperiksa.

## G. Langkah Selanjutnya

Lanjutkan ke [`LATIHAN_Panduan_Pemula.ipynb`](./LATIHAN_Panduan_Pemula.ipynb) untuk menerapkan keterampilan Open Data Cube, xarray, dan visualisasi dari panduan ini.